In [1]:
import sys
import mlflow
import pandas as pd
import requests

print("Python:", sys.version)
print("Python executable:", sys.executable)
print("MLflow:", mlflow.__version__)
print("Pandas:", pd.__version__)

# Verify Ollama
response = requests.get("http://localhost:11434/api/tags")
print("\nOllama status:", response.status_code)
print("Available models:", [
    model["name"] for model in response.json()["models"]
])

# Verify MLflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")
print("\nMLflow tracking URI:", mlflow.get_tracking_uri())

Python: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
Python executable: C:\Users\meetk\mlops-llmops-assignment2\.venv\Scripts\python.exe
MLflow: 3.18.0
Pandas: 3.1.1

Ollama status: 200
Available models: ['qwen2.5:1.5b']

MLflow tracking URI: http://127.0.0.1:5000


In [2]:

import mlflow
import pandas as pd

# Connect to our local MLflow server
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_registry_uri("http://127.0.0.1:5000")

# 1. Create the required experiment
EXPERIMENT_NAME = "eduassist_llmops_assignment"

experiment = mlflow.set_experiment(EXPERIMENT_NAME)

print("Experiment created/selected successfully")
print("Experiment name:", experiment.name)
print("Experiment ID:", experiment.experiment_id)

# 2. Define the three prompts exactly as required
PROMPT_NAME = "eduassist_academic_qa"

prompts = {
    "v1_basic": """Answer the following student question clearly.

Question: {{question}}""",

    "v2_structured": """You are an academic assistant.
Answer the following student question in a clear, concise, and accurate manner.
Use simple language suitable for undergraduate students.

Question: {{question}}""",

    "v3_json": """You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Do not include markdown, explanations outside JSON, or code fences.
Use this exact JSON schema:
{
  "answer": "string",
  "confidence": "high | medium | low",
  "key_points": ["string"]
}
Question: {{question}}"""
}

# 3. Register three sequential versions
commit_messages = {
    "v1_basic": "v1: Initial basic academic question-answering prompt",
    "v2_structured": "v2: Improved clarity and undergraduate-level answer structure",
    "v3_json": "v3: Added strict JSON response schema and output requirements"
}

registered_prompts = {}

for label, template in prompts.items():
    prompt = mlflow.genai.register_prompt(
        name=PROMPT_NAME,
        template=template,
        commit_message=commit_messages[label],
        tags={
            "assignment": "MLOps Assignment 2",
            "prompt_label": label,
            "model": "qwen2.5:1.5b"
        }
    )

    registered_prompts[label] = prompt

    print(
        f"{label}: Registered successfully | "
        f"Name: {prompt.name} | Version: {prompt.version}"
    )

# 4. Verify that the three registered versions can be loaded
print("\nVerifying prompt versions...")

for label, prompt in registered_prompts.items():
    loaded = mlflow.genai.load_prompt(
        f"prompts:/{PROMPT_NAME}/{prompt.version}",
        cache_ttl_seconds=0
    )

    test_prompt = loaded.format(
        question="What is machine learning?"
    )

    print(f"\n{label} | Version {prompt.version}")
    print(test_prompt[:250])

print("\nSTEP 8 COMPLETED")


2026/11/05 19:28:55 INFO mlflow.tracking.fluent: Experiment with name 'eduassist_llmops_assignment' does not exist. Creating a new experiment.


Experiment created/selected successfully
Experiment name: eduassist_llmops_assignment
Experiment ID: 1


2026/11/05 19:28:56 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: eduassist_academic_qa, version 1


v1_basic: Registered successfully | Name: eduassist_academic_qa | Version: 1


2026/11/05 19:28:57 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: eduassist_academic_qa, version 2
2026/11/05 19:28:57 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: eduassist_academic_qa, version 3


v2_structured: Registered successfully | Name: eduassist_academic_qa | Version: 2
v3_json: Registered successfully | Name: eduassist_academic_qa | Version: 3

Verifying prompt versions...

v1_basic | Version 1
Answer the following student question clearly.

Question: What is machine learning?

v2_structured | Version 2
You are an academic assistant.
Answer the following student question in a clear, concise, and accurate manner.
Use simple language suitable for undergraduate students.

Question: What is machine learning?

v3_json | Version 3
You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Do not include markdown, explanations outside JSON, or code fences.
Use this exact JSON schema:
{
  "answer": "string",
  "confidence": "high | medium | low",
  "k

STEP 8 COMPLETED


In [3]:

# Assignment 2: Five fixed academic questions

QUESTIONS = [
    {
        "question_id": "Q1",
        "question": "What is machine learning?"
    },
    {
        "question_id": "Q2",
        "question": "Explain supervised and unsupervised learning."
    },
    {
        "question_id": "Q3",
        "question": "What is overfitting, and how can it be prevented?"
    },
    {
        "question_id": "Q4",
        "question": "Explain the attention mechanism in transformers."
    },
    {
        "question_id": "Q5",
        "question": "What is the difference between precision and recall?"
    }
]

print("Total academic questions:", len(QUESTIONS))

for item in QUESTIONS:
    print(item["question_id"], ":", item["question"])

print("\nAll five questions are ready.")


Total academic questions: 5
Q1 : What is machine learning?
Q2 : Explain supervised and unsupervised learning.
Q3 : What is overfitting, and how can it be prevented?
Q4 : Explain the attention mechanism in transformers.
Q5 : What is the difference between precision and recall?

All five questions are ready.


In [4]:

import requests
import time
import mlflow

MODEL_NAME = "qwen2.5:1.5b"
OLLAMA_URL = "http://localhost:11434/api/generate"

PROMPT_NAME = "eduassist_academic_qa"

# Confirm MLflow connections
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_registry_uri("http://127.0.0.1:5000")

# Load registered prompt version 2 for warm-up
warmup_prompt = mlflow.genai.load_prompt(
    f"prompts:/{PROMPT_NAME}/2",
    cache_ttl_seconds=0
)

warmup_text = warmup_prompt.format(
    question="Reply with exactly the word READY."
)

# Warm up Ollama (excluded from experiment measurements)
warmup_response = requests.post(
    OLLAMA_URL,
    json={
        "model": MODEL_NAME,
        "prompt": warmup_text,
        "stream": False,
        "keep_alive": "10m",
        "options": {
            "temperature": 0.2,
            "num_predict": 16
        }
    },
    timeout=240
)

warmup_response.raise_for_status()
warmup_data = warmup_response.json()

print("Ollama API status:", warmup_response.status_code)
print("Model:", warmup_data["model"])
print("Warm-up response:", warmup_data["response"])
print("Model loaded successfully.")
print("Input tokens:", warmup_data["prompt_eval_count"])
print("Output tokens:", warmup_data["eval_count"])

print("\nWarm-up completed. This request is excluded from experiment metrics.")


Ollama API status: 200
Model: qwen2.5:1.5b
Warm-up response: READY
Model loaded successfully.
Input tokens: 67
Output tokens: 2

Warm-up completed. This request is excluded from experiment metrics.



## 3. LLM API Integration and Performance Measurement

We use the locally hosted Qwen 2.5 1.5B model through the Ollama REST API.

A reusable Python function is developed to execute prompts and capture:
- Request start and end timestamps
- Response latency
- Model name and HTTP status
- Input, output and total token counts
- Raw model response
- JSON validity against the required output schema

The same function is used to evaluate all three registered prompt versions.


In [5]:

import json
import time
import requests
import pandas as pd

from datetime import datetime, timezone

# Experiment configuration
MODEL_NAME = "qwen2.5:1.5b"
OLLAMA_URL = "http://localhost:11434/api/generate"
PROMPT_NAME = "eduassist_academic_qa"

# Validate the required JSON response structure
def validate_json_response(response_text):
    try:
        data = json.loads(response_text)

        required_fields = {"answer", "confidence", "key_points"}

        if not isinstance(data, dict):
            return False

        if set(data.keys()) != required_fields:
            return False

        if not isinstance(data["answer"], str):
            return False

        if data["confidence"] not in ["high", "medium", "low"]:
            return False

        if not isinstance(data["key_points"], list):
            return False

        if not all(isinstance(point, str) for point in data["key_points"]):
            return False

        return True

    except (json.JSONDecodeError, TypeError):
        return False


# Reusable function to execute one prompt request
def execute_llm_request(question_id, question, prompt_version):

    # Load the exact registered prompt version
    prompt = mlflow.genai.load_prompt(
        f"prompts:/{PROMPT_NAME}/{prompt_version}",
        cache_ttl_seconds=0
    )

    formatted_prompt = prompt.format(question=question)

    # Record request start time
    start_time = datetime.now(timezone.utc)
    timer_start = time.perf_counter()

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "prompt": formatted_prompt,
            "stream": False,
            "keep_alive": "10m",
            "options": {
                "temperature": 0.2,
                "num_predict": 256
            }
        },
        timeout=240
    )

    # Capture request completion time
    latency = time.perf_counter() - timer_start
    end_time = datetime.now(timezone.utc)

    response_status = response.status_code
    response.raise_for_status()

    result = response.json()

    answer = result.get("response", "").strip()

    input_tokens = result.get("prompt_eval_count", 0) or 0
    output_tokens = result.get("eval_count", 0) or 0

    # Validate JSON structure
    is_valid_json = validate_json_response(answer)

    record = {
        "question_id": question_id,
        "prompt_version": f"v{prompt_version}",
        "model_name": result.get("model", MODEL_NAME),
        "start_time": start_time.isoformat(),
        "end_time": end_time.isoformat(),
        "latency_seconds": round(latency, 6),
        "response_status": response_status,
        "response_state": "success",
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": input_tokens + output_tokens,
        "response_text": answer,
        "valid_json": is_valid_json,
        "ollama_total_duration_ns": result.get("total_duration", 0),
        "ollama_load_duration_ns": result.get("load_duration", 0)
    }

    return record


print("LLM execution function created.")
print("JSON validation function created.")


LLM execution function created.
JSON validation function created.


In [6]:

# Test one API request before the complete experiment

test_question = QUESTIONS[0]

test_result = execute_llm_request(
    question_id=test_question["question_id"],
    question=test_question["question"],
    prompt_version=1
)

print("TEST REQUEST RESULT")
print("-------------------")
print("Question:", test_result["question_id"])
print("Prompt version:", test_result["prompt_version"])
print("Model:", test_result["model_name"])
print("Latency:", test_result["latency_seconds"], "seconds")
print("HTTP status:", test_result["response_status"])
print("Input tokens:", test_result["input_tokens"])
print("Output tokens:", test_result["output_tokens"])
print("Total tokens:", test_result["total_tokens"])
print("Valid JSON:", test_result["valid_json"])

print("\nLLM RESPONSE:")
print(test_result["response_text"])

print("\nAPI FUNCTION TEST COMPLETED")


TEST REQUEST RESULT
-------------------
Question: Q1
Prompt version: v1
Model: qwen2.5:1.5b
Latency: 7.245745 seconds
HTTP status: 200
Input tokens: 43
Output tokens: 86
Total tokens: 129
Valid JSON: False

LLM RESPONSE:
Machine learning is a subfield of artificial intelligence that focuses on the development of algorithms and statistical models that allow computer systems to improve their performance on a specific task by learning from and making predictions based on data. It involves using statistical techniques to give computers the ability to learn without being explicitly programmed. Machine learning algorithms can be used for a wide range of tasks, including image and speech recognition, natural language processing, and predictive analytics.

API FUNCTION TEST COMPLETED


In [7]:

# Test the JSON-output prompt (v3) before the full experiment

json_test = execute_llm_request(
    question_id="Q1",
    question=QUESTIONS[0]["question"],
    prompt_version=3
)

print("JSON PROMPT VALIDATION TEST")
print("---------------------------")
print("Model:", json_test["model_name"])
print("HTTP status:", json_test["response_status"])
print("Latency:", json_test["latency_seconds"], "seconds")
print("Input tokens:", json_test["input_tokens"])
print("Output tokens:", json_test["output_tokens"])

print("\nMODEL RESPONSE:")
print(json_test["response_text"])

print("\nJSON VALIDATION:", json_test["valid_json"])

print("\nJSON PROMPT TEST COMPLETED")


JSON PROMPT VALIDATION TEST
---------------------------
Model: qwen2.5:1.5b
HTTP status: 200
Latency: 6.601193 seconds
Input tokens: 99
Output tokens: 74

MODEL RESPONSE:
{
  "answer": "Machine learning is a subset of artificial intelligence that involves the use of algorithms and statistical models to enable computers to learn from and make predictions on data without being explicitly programmed.",
  "confidence": "high",
  "key_points": ["subset of artificial intelligence", "learning from data", "predictions on data", "no explicit programming"]
}

JSON VALIDATION: True

JSON PROMPT TEST COMPLETED


## 4. Comparative Prompt Evaluation

The three registered prompt versions are evaluated against the same
five academic questions using Qwen 2.5 1.5B through the Ollama API.

All requests use the same model and generation configuration.
Latency, token consumption, HTTP status and JSON validity are captured
for each request.

The experiment consists of 15 measured API requests.

In [8]:

from pathlib import Path
import pandas as pd
import time

# Find the project root and prepare the output folder
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_FILE = OUTPUT_DIR / "latency_tokens_cost.csv"

# Map the MLflow versions to their experiment labels
PROMPT_VERSION_MAP = {
    1: "v1_basic",
    2: "v2_structured",
    3: "v3_json"
}

# Start with any previously saved results, if available
if RESULTS_FILE.exists():
    experiment_records = pd.read_csv(RESULTS_FILE).to_dict("records")
else:
    experiment_records = []

completed_requests = {
    (row["question_id"], row["prompt_version"])
    for row in experiment_records
    if row.get("response_state") == "success"
}

total_requests = len(QUESTIONS) * len(PROMPT_VERSION_MAP)

print("Starting LLMOps experiment")
print("Model:", MODEL_NAME)
print("Total requests:", total_requests)
print("Already completed:", len(completed_requests))

# Execute the five questions against each registered version
for version_number, version_label in PROMPT_VERSION_MAP.items():

    print(f"\n===== Evaluating {version_label} =====")

    for item in QUESTIONS:

        request_key = (item["question_id"], f"v{version_number}")

        # Avoid repeating requests already saved successfully
        if request_key in completed_requests:
            print(f"Skipping {request_key}: already completed")
            continue

        try:
            result = execute_llm_request(
                question_id=item["question_id"],
                question=item["question"],
                prompt_version=version_number
            )

            experiment_records.append(result)
            completed_requests.add(request_key)

            # Save immediately after every successful request
            pd.DataFrame(experiment_records).to_csv(
                RESULTS_FILE,
                index=False
            )

            print(
                f"{version_label} | {item['question_id']} | "
                f"Latency: {result['latency_seconds']:.2f}s | "
                f"Input: {result['input_tokens']} | "
                f"Output: {result['output_tokens']} | "
                f"JSON: {result['valid_json']}"
            )

        except Exception as error:
            print(
                f"ERROR | {version_label} | "
                f"{item['question_id']} | {error}"
            )

        # Small pause between requests
        time.sleep(1)

# Display the results
results_df = pd.DataFrame(experiment_records)

if not results_df.empty:
    results_df = results_df.sort_values(
        ["prompt_version", "question_id"]
    ).reset_index(drop=True)

    results_df.to_csv(RESULTS_FILE, index=False)

    display(
        results_df[
            [
                "question_id",
                "prompt_version",
                "latency_seconds",
                "response_status",
                "input_tokens",
                "output_tokens",
                "total_tokens",
                "valid_json"
            ]
        ]
    )

print("\nCompleted successful requests:", len(completed_requests))
print("Expected requests:", total_requests)
print("Results saved to:", RESULTS_FILE)


Starting LLMOps experiment
Model: qwen2.5:1.5b
Total requests: 15
Already completed: 0

===== Evaluating v1_basic =====
v1_basic | Q1 | Latency: 6.32s | Input: 43 | Output: 82 | JSON: False
v1_basic | Q2 | Latency: 16.64s | Input: 46 | Output: 256 | JSON: False
v1_basic | Q3 | Latency: 17.79s | Input: 51 | Output: 256 | JSON: False
v1_basic | Q4 | Latency: 16.84s | Input: 45 | Output: 256 | JSON: False
v1_basic | Q5 | Latency: 16.49s | Input: 47 | Output: 256 | JSON: False

===== Evaluating v2_structured =====
v2_structured | Q1 | Latency: 10.98s | Input: 65 | Output: 166 | JSON: False
v2_structured | Q2 | Latency: 12.65s | Input: 68 | Output: 207 | JSON: False
v2_structured | Q3 | Latency: 15.57s | Input: 73 | Output: 256 | JSON: False
v2_structured | Q4 | Latency: 12.60s | Input: 67 | Output: 197 | JSON: False
v2_structured | Q5 | Latency: 12.93s | Input: 69 | Output: 203 | JSON: False

===== Evaluating v3_json =====
v3_json | Q1 | Latency: 5.66s | Input: 99 | Output: 71 | JSON: True

,question_id,prompt_version,latency_seconds,response_status,input_tokens,output_tokens,total_tokens,valid_json
0,Q1,v1,6.320499,200,43,82,125,False
1,Q2,v1,16.639979,200,46,256,302,False
2,Q3,v1,17.790404,200,51,256,307,False
3,Q4,v1,16.838274,200,45,256,301,False
4,Q5,v1,16.488754,200,47,256,303,False
5,Q1,v2,10.977016,200,65,166,231,False
6,Q2,v2,12.649437,200,68,207,275,False
7,Q3,v2,15.573143,200,73,256,329,False
8,Q4,v2,12.604450,200,67,197,264,False
9,Q5,v2,12.934604,200,69,203,272,False



Completed successful requests: 15
Expected requests: 15
Results saved to: C:\Users\meetk\mlops-llmops-assignment2\outputs\latency_tokens_cost.csv


In [9]:

from pathlib import Path
import pandas as pd
import time
import requests
import mlflow
from datetime import datetime, timezone

# 1. Preserve the initial 256-token experiment
OUTPUT_DIR = Path.cwd()
if OUTPUT_DIR.name.lower() == "notebooks":
    OUTPUT_DIR = OUTPUT_DIR.parent

OUTPUT_DIR = OUTPUT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_FILE = OUTPUT_DIR / "latency_tokens_cost.csv"
INITIAL_FILE = OUTPUT_DIR / "latency_tokens_cost_initial_256.csv"

pd.DataFrame(experiment_records).to_csv(INITIAL_FILE, index=False)
print("Initial results preserved:", INITIAL_FILE)

# 2. Update the request function with a higher output limit
def execute_llm_request(question_id, question, prompt_version):

    prompt = mlflow.genai.load_prompt(
        f"prompts:/{PROMPT_NAME}/{prompt_version}",
        cache_ttl_seconds=0
    )

    formatted_prompt = prompt.format(question=question)

    start_time = datetime.now(timezone.utc)
    timer_start = time.perf_counter()

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "prompt": formatted_prompt,
            "stream": False,
            "keep_alive": "10m",
            "options": {
                "temperature": 0.2,
                "num_predict": 512
            }
        },
        timeout=300
    )

    latency = time.perf_counter() - timer_start
    end_time = datetime.now(timezone.utc)

    response.raise_for_status()
    result = response.json()

    answer = result.get("response", "").strip()
    input_tokens = result.get("prompt_eval_count", 0) or 0
    output_tokens = result.get("eval_count", 0) or 0

    return {
        "question_id": question_id,
        "prompt_version": f"v{prompt_version}",
        "model_name": result.get("model", MODEL_NAME),
        "start_time": start_time.isoformat(),
        "end_time": end_time.isoformat(),
        "latency_seconds": round(latency, 6),
        "response_status": response.status_code,
        "response_state": "success",
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": input_tokens + output_tokens,
        "response_text": answer,
        "valid_json": validate_json_response(answer),
        "done_reason": result.get("done_reason"),
        "ollama_total_duration_ns": result.get("total_duration", 0),
        "ollama_load_duration_ns": result.get("load_duration", 0)
    }


# 3. Run all 15 requests again with identical settings
experiment_records = []

for version_number, version_label in {
    1: "v1_basic",
    2: "v2_structured",
    3: "v3_json"
}.items():

    print(f"\n===== {version_label} =====")

    for item in QUESTIONS:

        result = execute_llm_request(
            item["question_id"],
            item["question"],
            version_number
        )

        experiment_records.append(result)

        # Save after every request
        pd.DataFrame(experiment_records).to_csv(
            RESULTS_FILE, index=False
        )

        print(
            f"{result['prompt_version']} | {result['question_id']} | "
            f"Latency: {result['latency_seconds']:.2f}s | "
            f"Output tokens: {result['output_tokens']} | "
            f"JSON: {result['valid_json']} | "
            f"Stop reason: {result['done_reason']}"
        )

        time.sleep(1)

# 4. Show and save the final experiment results
results_df = pd.DataFrame(experiment_records)

display(
    results_df[
        [
            "question_id",
            "prompt_version",
            "latency_seconds",
            "response_status",
            "input_tokens",
            "output_tokens",
            "total_tokens",
            "valid_json",
            "done_reason"
        ]
    ]
)

print("\nFinal requests:", len(results_df))
print("Successful requests:", (results_df["response_status"] == 200).sum())
print("Final CSV:", RESULTS_FILE)


Initial results preserved: C:\Users\meetk\mlops-llmops-assignment2\outputs\latency_tokens_cost_initial_256.csv

===== v1_basic =====
v1 | Q1 | Latency: 7.47s | Output tokens: 85 | JSON: False | Stop reason: stop
v1 | Q2 | Latency: 12.71s | Output tokens: 205 | JSON: False | Stop reason: stop
v1 | Q3 | Latency: 16.56s | Output tokens: 283 | JSON: False | Stop reason: stop
v1 | Q4 | Latency: 29.34s | Output tokens: 512 | JSON: False | Stop reason: length
v1 | Q5 | Latency: 13.70s | Output tokens: 226 | JSON: False | Stop reason: stop

===== v2_structured =====
v2 | Q1 | Latency: 18.31s | Output tokens: 318 | JSON: False | Stop reason: stop
v2 | Q2 | Latency: 15.06s | Output tokens: 254 | JSON: False | Stop reason: stop
v2 | Q3 | Latency: 19.72s | Output tokens: 335 | JSON: False | Stop reason: stop
v2 | Q4 | Latency: 14.29s | Output tokens: 229 | JSON: False | Stop reason: stop
v2 | Q5 | Latency: 12.34s | Output tokens: 176 | JSON: False | Stop reason: stop

===== v3_json =====
v3 | Q1 |

,question_id,prompt_version,latency_seconds,response_status,input_tokens,output_tokens,total_tokens,valid_json,done_reason
0,Q1,v1,7.474017,200,43,85,128,False,stop
1,Q2,v1,12.710916,200,46,205,251,False,stop
2,Q3,v1,16.561850,200,51,283,334,False,stop
3,Q4,v1,29.344616,200,45,512,557,False,length
4,Q5,v1,13.696120,200,47,226,273,False,stop
5,Q1,v2,18.307340,200,65,318,383,False,stop
6,Q2,v2,15.057105,200,68,254,322,False,stop
7,Q3,v2,19.722622,200,73,335,408,False,stop
8,Q4,v2,14.293786,200,67,229,296,False,stop
9,Q5,v2,12.340599,200,69,176,245,False,stop



Final requests: 15
Successful requests: 15
Final CSV: C:\Users\meetk\mlops-llmops-assignment2\outputs\latency_tokens_cost.csv


## 5. Latency Analysis and Token-Based Cost Estimation

Latency statistics are calculated separately for each prompt version.
Input and output token counts are extracted from Ollama API responses.

Hosted inference costs are estimated using the published
Qwen2.5-1.5B-Instruct pricing from Featherless AI.

Local Ollama inference does not incur per-request API charges.
The hosted cost is included as a comparable economic estimate.

In [10]:

# Task 1 and Task 2: Performance and cost analysis

import pandas as pd
from pathlib import Path

# Published hosted-equivalent pricing (USD per million tokens)
INPUT_PRICE = 0.04
OUTPUT_PRICE = 0.08
MONTHLY_REQUESTS = 10000

PRICING_PROVIDER = "Featherless AI"
PRICING_DATE = "2026-11-05"

# Load the final 15-request experiment
results_df = pd.DataFrame(experiment_records).copy()

assert len(results_df) == 15, "Expected exactly 15 measured requests."
assert (results_df["response_status"] == 200).all()

# Calculate input and output costs separately
results_df["input_cost_usd"] = (
    results_df["input_tokens"] / 1_000_000
) * INPUT_PRICE

results_df["output_cost_usd"] = (
    results_df["output_tokens"] / 1_000_000
) * OUTPUT_PRICE

results_df["estimated_hosted_cost_usd"] = (
    results_df["input_cost_usd"] +
    results_df["output_cost_usd"]
)

results_df["local_ollama_api_cost_usd"] = 0.0

# Add pricing information for transparency
results_df["pricing_provider"] = PRICING_PROVIDER
results_df["pricing_date"] = PRICING_DATE
results_df["currency"] = "USD"

# Task 1: Latency statistics per prompt version
latency_stats = (
    results_df.groupby("prompt_version")["latency_seconds"]
    .agg(
        mean="mean",
        median="median",
        minimum="min",
        maximum="max",
        standard_deviation="std"
    )
    .reset_index()
)

# Task 1: Overall latency statistics
overall_stats = pd.DataFrame([{
    "prompt_version": "overall",
    "mean": results_df["latency_seconds"].mean(),
    "median": results_df["latency_seconds"].median(),
    "minimum": results_df["latency_seconds"].min(),
    "maximum": results_df["latency_seconds"].max(),
    "standard_deviation": results_df["latency_seconds"].std()
}])

latency_stats = pd.concat(
    [latency_stats, overall_stats],
    ignore_index=True
)

# Task 3 comparison metrics
prompt_comparison = (
    results_df.groupby("prompt_version")
    .agg(
        avg_latency_seconds=("latency_seconds", "mean"),
        avg_input_tokens=("input_tokens", "mean"),
        avg_output_tokens=("output_tokens", "mean"),
        avg_total_tokens=("total_tokens", "mean"),
        avg_cost_per_request=("estimated_hosted_cost_usd", "mean"),
        valid_json=("valid_json", "mean")
    )
    .reset_index()
)

# Project monthly cost at 10,000 requests per month
prompt_comparison["monthly_projection_10000_usd"] = (
    prompt_comparison["avg_cost_per_request"] * MONTHLY_REQUESTS
)

# Print latency statistics
print("LATENCY STATISTICS")
display(latency_stats.round(4))

# Print request-level cost data
print("\nREQUEST-LEVEL TOKEN AND COST DATA")
display(
    results_df[
        [
            "question_id",
            "prompt_version",
            "input_tokens",
            "output_tokens",
            "total_tokens",
            "input_cost_usd",
            "output_cost_usd",
            "estimated_hosted_cost_usd"
        ]
    ].round(9)
)

# Print prompt comparison
print("\nPROMPT COMPARISON")
display(prompt_comparison.round(6))

# Overall monthly cost projection
average_request_cost = results_df[
    "estimated_hosted_cost_usd"
].mean()

monthly_projection = average_request_cost * MONTHLY_REQUESTS

print("\nOVERALL MONTHLY COST PROJECTION")
print("Average hosted cost per request: $", round(average_request_cost, 9))
print("Requests per month:", MONTHLY_REQUESTS)
print("Projected monthly hosted cost: $", round(monthly_projection, 6))
print("Actual local Ollama API cost per request: $0.00")

# Save all results
OUTPUT_DIR = Path.cwd()
if OUTPUT_DIR.name.lower() == "notebooks":
    OUTPUT_DIR = OUTPUT_DIR.parent
OUTPUT_DIR = OUTPUT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(
    OUTPUT_DIR / "latency_tokens_cost.csv", index=False
)

latency_stats.to_csv(
    OUTPUT_DIR / "latency_statistics.csv", index=False
)

prompt_comparison.to_csv(
    OUTPUT_DIR / "prompt_comparison.csv", index=False
)

print("\nAll analysis CSV files saved successfully.")


LATENCY STATISTICS


,prompt_version,mean,median,minimum,maximum,standard_deviation
0,v1,15.9575,13.6961,7.4740,29.3446,8.1732
1,v2,15.9443,15.0571,12.3406,19.7226,3.0149
2,v3,13.8062,10.7143,5.9820,29.8436,9.3181
3,overall,15.2360,13.6961,5.9820,29.8436,6.8982



REQUEST-LEVEL TOKEN AND COST DATA


,question_id,prompt_version,input_tokens,output_tokens,total_tokens,input_cost_usd,output_cost_usd,estimated_hosted_cost_usd
0,Q1,v1,43,85,128,0.000002,0.000007,0.000009
1,Q2,v1,46,205,251,0.000002,0.000016,0.000018
2,Q3,v1,51,283,334,0.000002,0.000023,0.000025
3,Q4,v1,45,512,557,0.000002,0.000041,0.000043
4,Q5,v1,47,226,273,0.000002,0.000018,0.000020
5,Q1,v2,65,318,383,0.000003,0.000025,0.000028
6,Q2,v2,68,254,322,0.000003,0.000020,0.000023
7,Q3,v2,73,335,408,0.000003,0.000027,0.000030
8,Q4,v2,67,229,296,0.000003,0.000018,0.000021
9,Q5,v2,69,176,245,0.000003,0.000014,0.000017



PROMPT COMPARISON


,prompt_version,avg_latency_seconds,avg_input_tokens,avg_output_tokens,avg_total_tokens,avg_cost_per_request,valid_json,monthly_projection_10000_usd
0,v1,15.957504,46.4,262.2,308.6,0.000023,0.0,0.22832
1,v2,15.944290,68.4,262.4,330.8,0.000024,0.0,0.23728
2,v3,13.806184,102.4,222.4,324.8,0.000022,0.8,0.21888



OVERALL MONTHLY COST PROJECTION
Average hosted cost per request: $ 2.2816e-05
Requests per month: 10000
Projected monthly hosted cost: $ 0.22816
Actual local Ollama API cost per request: $0.00

All analysis CSV files saved successfully.


## MLflow Prompt Evaluation and Comparison

Each registered prompt version is evaluated using the same five academic
questions and the same Qwen 2.5 1.5B model.

The average latency, input tokens, output tokens, estimated hosted cost,
and JSON validity rate are aggregated and logged as MLflow run metrics.

Each run is associated with its corresponding registered prompt version.

In [11]:

# Task 3: Log prompt evaluation metrics in MLflow

import mlflow
import pandas as pd
from pathlib import Path

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_registry_uri("http://127.0.0.1:5000")

mlflow.set_experiment("eduassist_llmops_assignment")

OUTPUT_DIR = Path.cwd()
if OUTPUT_DIR.name.lower() == "notebooks":
    OUTPUT_DIR = OUTPUT_DIR.parent

OUTPUT_DIR = OUTPUT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

version_labels = {
    "v1": "v1_basic",
    "v2": "v2_structured",
    "v3": "v3_json"
}

mlflow_runs = []

for version, label in version_labels.items():

    subset = results_df[
        results_df["prompt_version"] == version
    ].copy()

    if len(subset) != 5:
        raise ValueError(f"{version} must contain exactly five requests.")

    # Calculate the required evaluation metrics
    metrics = {
        "avg_latency_seconds": float(
            subset["latency_seconds"].mean()
        ),
        "avg_input_tokens": float(
            subset["input_tokens"].mean()
        ),
        "avg_output_tokens": float(
            subset["output_tokens"].mean()
        ),
        "avg_cost_per_request": float(
            subset["estimated_hosted_cost_usd"].mean()
        ),
        "valid_json": float(
            subset["valid_json"].astype(float).mean()
        )
    }

    prompt = registered_prompts[label]

    with mlflow.start_run(run_name=f"Prompt_{version}") as run:

        # Log experiment parameters
        mlflow.log_params({
            "model_name": MODEL_NAME,
            "prompt_registry_name": PROMPT_NAME,
            "prompt_version": version,
            "prompt_registry_version": str(prompt.version),
            "inference_backend": "Local Ollama",
            "generation_temperature": "0.2",
            "max_output_tokens": "512",
            "cost_estimation_provider": "Featherless AI"
        })

        # Log the five required assignment metrics
        mlflow.log_metrics(metrics)

        # Log additional evaluation information
        mlflow.log_metrics({
            "total_requests": 5.0,
            "json_valid_responses": float(
                subset["valid_json"].sum()
            ),
            "length_truncated_responses": float(
                (subset["done_reason"] == "length").sum()
            )
        })

        # Log the registered prompt template as a run artifact
        mlflow.log_text(
            prompts[label],
            "prompt_template.txt"
        )

        # Save request-level results for this version
        version_file = OUTPUT_DIR / f"evaluation_{version}.csv"
        subset.to_csv(version_file, index=False)
        mlflow.log_artifact(str(version_file))

        mlflow.set_tags({
            "prompt_label": label,
            "prompt_registry": PROMPT_NAME,
            "assignment": "MLOps Assignment 2"
        })

        mlflow_runs.append({
            "prompt_version": version,
            "run_id": run.info.run_id,
            **metrics
        })

        print(f"{version} logged successfully.")
        print("Run ID:", run.info.run_id)
        print("Metrics:", metrics)
        print("-" * 50)


# Generate the final prompt evaluation CSV
prompt_evaluation_df = pd.DataFrame(mlflow_runs)

prompt_evaluation_df.to_csv(
    OUTPUT_DIR / "prompt_evaluation_results.csv",
    index=False
)

print("\nALL THREE PROMPT VERSIONS LOGGED SUCCESSFULLY")
display(prompt_evaluation_df)


v1 logged successfully.
Run ID: d14ea92a669a47c099d4e830a4d063ae
Metrics: {'avg_latency_seconds': 15.957503800000001, 'avg_input_tokens': 46.4, 'avg_output_tokens': 262.2, 'avg_cost_per_request': 2.2832e-05, 'valid_json': 0.0}
--------------------------------------------------
🏃 View run Prompt_v1 at: http://127.0.0.1:5000/#/experiments/1/runs/d14ea92a669a47c099d4e830a4d063ae
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
v2 logged successfully.
Run ID: a0593193de57432283e6fb4f8c2bc0c7
Metrics: {'avg_latency_seconds': 15.9442904, 'avg_input_tokens': 68.4, 'avg_output_tokens': 262.4, 'avg_cost_per_request': 2.3728000000000004e-05, 'valid_json': 0.0}
--------------------------------------------------
🏃 View run Prompt_v2 at: http://127.0.0.1:5000/#/experiments/1/runs/a0593193de57432283e6fb4f8c2bc0c7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
v3 logged successfully.
Run ID: cb2bb119a51a410b9688596f7c1790f9
Metrics: {'avg_latency_seconds': 13.806183999999998, 

,prompt_version,run_id,avg_latency_seconds,avg_input_tokens,avg_output_tokens,avg_cost_per_request,valid_json
0,v1,d14ea92a669a47c099d4e830a4d063ae,15.957504,46.4,262.2,0.000023,0.0
1,v2,a0593193de57432283e6fb4f8c2bc0c7,15.944290,68.4,262.4,0.000024,0.0
2,v3,cb2bb119a51a410b9688596f7c1790f9,13.806184,102.4,222.4,0.000022,0.8


In [12]:

import difflib
from pathlib import Path
import pandas as pd

# Build comparison using actual experimental results
comparison = prompt_comparison.copy()

comparison["valid_json"] = (
    comparison["valid_json"] * 100
).round(2)

comparison["avg_latency_seconds"] = (
    comparison["avg_latency_seconds"].round(4)
)

comparison["avg_input_tokens"] = (
    comparison["avg_input_tokens"].round(2)
)

comparison["avg_output_tokens"] = (
    comparison["avg_output_tokens"].round(2)
)

comparison["avg_cost_per_request"] = (
    comparison["avg_cost_per_request"].map(
        lambda x: f"${x:.10f}"
    )
)

comparison["recommended"] = comparison[
    "prompt_version"
].eq("v3")

print("FINAL PROMPT COMPARISON")
display(comparison)

# Generate prompt differences
diff_parts = []

prompt_pairs = [
    ("v1_basic", "v2_structured"),
    ("v2_structured", "v3_json")
]

for old_label, new_label in prompt_pairs:

    diff = difflib.unified_diff(
        prompts[old_label].splitlines(),
        prompts[new_label].splitlines(),
        fromfile=old_label,
        tofile=new_label,
        lineterm=""
    )

    diff_parts.append(
        f"\nPROMPT COMPARISON: {old_label} -> {new_label}\n"
        + "\n".join(diff)
    )

prompt_diff_text = "\n".join(diff_parts)

# Save comparison and difference evidence
OUTPUT_DIR = Path.cwd()
if OUTPUT_DIR.name.lower() == "notebooks":
    OUTPUT_DIR = OUTPUT_DIR.parent
OUTPUT_DIR = OUTPUT_DIR / "outputs"

comparison.to_csv(
    OUTPUT_DIR / "prompt_comparison.csv",
    index=False
)

(OUTPUT_DIR / "prompt_diff.txt").write_text(
    prompt_diff_text,
    encoding="utf-8"
)

print("\nPrompt comparison CSV saved.")
print("Prompt difference file saved.")

print("\nPROMPT DIFFERENCES:")
print(prompt_diff_text)


FINAL PROMPT COMPARISON


,prompt_version,avg_latency_seconds,avg_input_tokens,avg_output_tokens,avg_total_tokens,avg_cost_per_request,valid_json,monthly_projection_10000_usd,recommended
0,v1,15.9575,46.4,262.2,308.6,$0.0000228320,0.0,0.22832,False
1,v2,15.9443,68.4,262.4,330.8,$0.0000237280,0.0,0.23728,False
2,v3,13.8062,102.4,222.4,324.8,$0.0000218880,80.0,0.21888,True



Prompt comparison CSV saved.
Prompt difference file saved.

PROMPT DIFFERENCES:

PROMPT COMPARISON: v1_basic -> v2_structured
--- v1_basic
+++ v2_structured
@@ -1,3 +1,5 @@
-Answer the following student question clearly.
+You are an academic assistant.
+Answer the following student question in a clear, concise, and accurate manner.
+Use simple language suitable for undergraduate students.
 
 Question: {{question}}

PROMPT COMPARISON: v2_structured -> v3_json
--- v2_structured
+++ v3_json
@@ -1,5 +1,10 @@
 You are an academic assistant.
-Answer the following student question in a clear, concise, and accurate manner.
-Use simple language suitable for undergraduate students.
-
+Answer the student question and return ONLY valid JSON.
+Do not include markdown, explanations outside JSON, or code fences.
+Use this exact JSON schema:
+{
+  "answer": "string",
+  "confidence": "high | medium | low",
+  "key_points": ["string"]
+}
 Question: {{question}}


In [13]:

# Promote the evaluated best prompt version to production

import mlflow

PROMPT_NAME = "eduassist_academic_qa"
PRODUCTION_VERSION = 3

# Assign the production alias
mlflow.genai.set_prompt_alias(
    name=PROMPT_NAME,
    alias="production",
    version=PRODUCTION_VERSION
)

# Load the prompt through its production alias
production_prompt = mlflow.genai.load_prompt(
    f"prompts:/{PROMPT_NAME}@production",
    cache_ttl_seconds=0
)

# Verify the assigned version
assert str(production_prompt.version) == str(PRODUCTION_VERSION)

print("PROMPT PROMOTION SUCCESSFUL")
print("Prompt:", production_prompt.name)
print("Production version:", production_prompt.version)
print("Alias: production")
print("\nProduction prompt verified successfully.")


PROMPT PROMOTION SUCCESSFUL
Prompt: eduassist_academic_qa
Production version: 3
Alias: production

Production prompt verified successfully.


## Prompt Promotion Justification

Prompt Version 3 is recommended for production because it provides the best
overall operational fit for the EduAssist AI application. Version 1 uses the
lowest average input token count, but it does not follow the required JSON
response structure. Version 2 provides more explicit academic instructions,
but it also failed the JSON validation checks in all five requests.

Version 3 achieved an 80% JSON validity rate, with four out of five responses
passing the structural validation. It also recorded the lowest average
latency of approximately 13.81 seconds, compared with 15.96 seconds for
Version 1 and 15.94 seconds for Version 2. Its average output consumption
was 222.4 tokens, lower than the other two versions. Its estimated hosted
inference cost was approximately $0.000021888 per request, also the lowest
among the three versions.

However, Version 3 did produce one invalid JSON response for Question 2
because generation reached the configured 512-token limit. This indicates
that output truncation and response-length management require further
attention before a real production deployment.

Despite this limitation, Version 3 is the most suitable candidate because
it best meets the application's strict structured-output requirement while
providing favourable latency and cost measurements. It is therefore
promoted using the MLflow production alias, with additional JSON validation
and truncation handling recommended before live deployment.

In [14]:
# Correct and verify monthly projections

prompt_comparison["monthly_projection_10000_usd"] = (
    prompt_comparison["avg_cost_per_request"] * 10000
)

display(
    prompt_comparison[
        [
            "prompt_version",
            "avg_latency_seconds",
            "avg_input_tokens",
            "avg_output_tokens",
            "avg_cost_per_request",
            "valid_json",
            "monthly_projection_10000_usd"
        ]
    ].round(8)
)

prompt_comparison.to_csv(
    OUTPUT_DIR / "prompt_comparison.csv",
    index=False
)

,prompt_version,avg_latency_seconds,avg_input_tokens,avg_output_tokens,avg_cost_per_request,valid_json,monthly_projection_10000_usd
0,v1,15.957504,46.4,262.2,0.000023,0.0,0.22832
1,v2,15.944290,68.4,262.4,0.000024,0.0,0.23728
2,v3,13.806184,102.4,222.4,0.000022,0.8,0.21888


In [15]:

# Recalculate costs using the official published Qwen2.5-1.5B model-class rates

from mlflow.tracking import MlflowClient

INPUT_PRICE = 0.01
OUTPUT_PRICE = 0.02
MONTHLY_REQUESTS = 10000

# Recalculate request-level costs
results_df["input_cost_usd"] = (
    results_df["input_tokens"] / 1_000_000
) * INPUT_PRICE

results_df["output_cost_usd"] = (
    results_df["output_tokens"] / 1_000_000
) * OUTPUT_PRICE

results_df["estimated_hosted_cost_usd"] = (
    results_df["input_cost_usd"] + results_df["output_cost_usd"]
)

results_df["local_ollama_api_cost_usd"] = 0.0
results_df["pricing_provider"] = "Featherless AI"
results_df["pricing_date"] = "2026-11-05"
results_df["currency"] = "USD"

# Rebuild prompt comparison from corrected request costs
prompt_comparison = (
    results_df.groupby("prompt_version")
    .agg(
        avg_latency_seconds=("latency_seconds", "mean"),
        avg_input_tokens=("input_tokens", "mean"),
        avg_output_tokens=("output_tokens", "mean"),
        avg_total_tokens=("total_tokens", "mean"),
        avg_cost_per_request=("estimated_hosted_cost_usd", "mean"),
        valid_json=("valid_json", "mean")
    )
    .reset_index()
)

prompt_comparison["monthly_projection_10000_usd"] = (
    prompt_comparison["avg_cost_per_request"] * MONTHLY_REQUESTS
)

# Update the existing MLflow evaluation runs without creating duplicates
client = MlflowClient()

for item in mlflow_runs:
    version = item["prompt_version"]

    new_cost = prompt_comparison.loc[
        prompt_comparison["prompt_version"] == version,
        "avg_cost_per_request"
    ].iloc[0]

    client.log_metric(
        run_id=item["run_id"],
        key="avg_cost_per_request",
        value=float(new_cost)
    )

    item["avg_cost_per_request"] = float(new_cost)

# Save corrected deliverables
OUTPUT_DIR = Path.cwd()
if OUTPUT_DIR.name.lower() == "notebooks":
    OUTPUT_DIR = OUTPUT_DIR.parent

OUTPUT_DIR = OUTPUT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(
    OUTPUT_DIR / "latency_tokens_cost.csv", index=False
)

prompt_comparison.to_csv(
    OUTPUT_DIR / "prompt_comparison.csv", index=False
)

prompt_evaluation_df = pd.DataFrame(mlflow_runs)
prompt_evaluation_df.to_csv(
    OUTPUT_DIR / "prompt_evaluation_results.csv", index=False
)

print("CORRECTED PRICING")
print("Input price: $0.01 per million tokens")
print("Output price: $0.02 per million tokens")

display(
    prompt_comparison[
        [
            "prompt_version",
            "avg_latency_seconds",
            "avg_input_tokens",
            "avg_output_tokens",
            "avg_cost_per_request",
            "valid_json",
            "monthly_projection_10000_usd"
        ]
    ].round(9)
)

print("\nCorrected CSV files saved.")
print("Existing MLflow runs updated; no duplicate runs created.")


CORRECTED PRICING
Input price: $0.01 per million tokens
Output price: $0.02 per million tokens


,prompt_version,avg_latency_seconds,avg_input_tokens,avg_output_tokens,avg_cost_per_request,valid_json,monthly_projection_10000_usd
0,v1,15.957504,46.4,262.2,0.000006,0.0,0.05708
1,v2,15.944290,68.4,262.4,0.000006,0.0,0.05932
2,v3,13.806184,102.4,222.4,0.000005,0.8,0.05472



Corrected CSV files saved.
Existing MLflow runs updated; no duplicate runs created.


##  Findings, Interpretation and Conclusion

### Latency interpretation

The slowest request was Q2 using Prompt Version 3, with a latency
of approximately 29.84 seconds. This request generated 512 output
tokens and stopped because it reached the configured generation limit.
The longer generation length contributed to the increased latency.
Local hardware resources and model generation speed may also influence
response time.

### Cost optimisation

Prompt Version 3 achieved the lowest average estimated hosted cost
because it generated fewer output tokens on average. Further savings
could be obtained by limiting response length, removing unnecessary
context and caching stable prompt content.

### JSON validity

Prompt Version 3 achieved an 80% JSON validity rate, while Versions 1
and 2 achieved 0%. One Version 3 response reached the output-token
limit, demonstrating the importance of truncation handling and
post-generation schema validation.

### Conclusion

The experiment demonstrates how prompt versions can be evaluated
using operational metrics and managed through MLflow. Version 3 was
selected and assigned the production alias based on its JSON validity,
latency and estimated cost results. The remaining JSON failure is
documented as a limitation requiring further improvement.

In [16]:

from pathlib import Path
from importlib.metadata import version

# Locate the project root
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name.lower() == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

# 1. Generate requirements.txt using installed package versions
packages = [
    ("mlflow[genai]", "mlflow"),
    ("pandas", "pandas"),
    ("requests", "requests"),
    ("jupyterlab", "jupyterlab"),
    ("ipykernel", "ipykernel"),
    ("nbconvert", "nbconvert"),
    ("jsonschema", "jsonschema")
]

requirements = []

for package_name, distribution_name in packages:
    installed_version = version(distribution_name)
    requirements.append(f"{package_name}=={installed_version}")

(PROJECT_DIR / "requirements.txt").write_text(
    "\n".join(requirements) + "\n",
    encoding="utf-8"
)

# 2. Generate the project README
readme_content = """
# EduAssist AI - LLMOps Assignment 2

## 1. Project Overview

This project evaluates three academic question-answering prompt
versions using a locally hosted Qwen 2.5 1.5B model.

The experiment measures response latency, token consumption,
estimated hosted inference cost and JSON output validity.

MLflow is used for prompt registration, evaluation tracking,
comparison and production alias management.

## 2. Technology Stack

- Python 3.13.9
- Ollama 0.40.14
- Qwen 2.5 1.5B (`qwen2.5:1.5b`)
- MLflow 3.18.0
- SQLite
- JupyterLab
- Pandas
- Requests
- JSON Schema validation

## 3. Model and API Configuration

Inference provider: Local Ollama

Model: qwen2.5:1.5b

API version/endpoint: Ollama REST API `/api/generate`

Base URL: http://localhost:11434

The model runs locally. No cloud API key is required.

Token counts are obtained from Ollama response fields:
- prompt_eval_count
- eval_count

## 4. Pricing Methodology

Local Ollama inference has no per-request cloud API charge.

For economic comparison, estimated hosted inference cost is
calculated using the Featherless Qwen 2.5 1.5B model-class pricing:

- Input: USD 0.01 per million tokens
- Output: USD 0.02 per million tokens
- Currency: USD
- Pricing reference date: 2026-11-05
- Pricing source:
  https://featherless.ai/docs/request-pricing-and-credits

These values represent estimated equivalent hosted inference
charges and not actual local Ollama expenditure.

## 5. Experiment Details

MLflow experiment:
`eduassist_llmops_assignment`

Registered prompt:
`eduassist_academic_qa`

Prompt versions:
- Version 1: Basic
- Version 2: Structured
- Version 3: JSON Output

Five academic questions were evaluated against each version,
resulting in 15 measured requests.

The production alias was assigned to Version 3 based on its
measured performance and structured-output compliance.

## 6. Installation

Install Ollama for Windows from:

https://ollama.com/download/windows

Download the model:

```powershell
ollama pull qwen2.5:1.5b
```

Create and activate a Python environment:

```powershell
python -m venv .venv
Set-ExecutionPolicy -Scope Process -ExecutionPolicy Bypass -Force
.\\.venv\\Scripts\\Activate.ps1
python -m pip install -r requirements.txt
```

## 7. Start MLflow

From the project root, run:

```powershell
$ArtifactPath = Join-Path (Get-Location).Path "mlartifacts"

mlflow server `
  --backend-store-uri "sqlite:///mlflow.db" `
  --artifacts-destination "$ArtifactPath" `
  --host 127.0.0.1 `
  --port 5000
```

MLflow UI:

http://127.0.0.1:5000

## 8. Environment Variables

The experiment uses these local service settings:

- MLFLOW_TRACKING_URI=http://127.0.0.1:5000
- MLFLOW_REGISTRY_URI=http://127.0.0.1:5000
- OLLAMA_URL=http://localhost:11434/api/generate
- MODEL_NAME=qwen2.5:1.5b

No API key is required for the local Ollama fallback.

## 9. Reproduction Steps

1. Start Ollama and ensure the Qwen model is available.
2. Start the MLflow tracking server.
3. Activate the Python virtual environment.
4. Launch JupyterLab and open `notebooks/LLMOps_Assignment2.ipynb`.
5. Execute the notebook cells in order.
6. Review the three registered prompt versions and evaluation runs.
7. Review the CSV outputs in the `outputs` directory.
8. Open MLflow Prompts and verify Version 3 has the production alias.

## 10. Output Files

- `outputs/latency_tokens_cost.csv`
- `outputs/latency_statistics.csv`
- `outputs/prompt_comparison.csv`
- `outputs/prompt_evaluation_results.csv`
- `screenshots/`
- `reports/`

## 11. Observations and Limitations

Version 3 achieved an 80% JSON validity rate. One response
reached the configured generation limit, demonstrating the need
for output-length management and post-generation validation.

The cost estimates use published hosted inference rates.
They are not actual charges for local inference.
"""

(PROJECT_DIR / "README.md").write_text(
    readme_content.strip() + "\n",
    encoding="utf-8"
)

print("Project files created successfully.")
print("\nREQUIREMENTS:")
print((PROJECT_DIR / "requirements.txt").read_text())

print("\nREADME created:", PROJECT_DIR / "README.md")
print("Project root:", PROJECT_DIR)


Project files created successfully.

REQUIREMENTS:
mlflow[genai]==3.18.0
pandas==3.1.1
requests==2.34.2
jupyterlab==4.6.5
ipykernel==7.4.0
nbconvert==7.17.2
jsonschema==4.26.0


README created: C:\Users\meetk\mlops-llmops-assignment2\README.md
Project root: C:\Users\meetk\mlops-llmops-assignment2


In [17]:
from pathlib import Path

for item in sorted(PROJECT_DIR.iterdir()):
    print(item.name, " - ", "Directory" if item.is_dir() else "File")

print("\nRequirements file exists:",
      (PROJECT_DIR / "requirements.txt").exists())

print("README file exists:",
      (PROJECT_DIR / "README.md").exists())

print("Notebook exists:",
      (PROJECT_DIR / "notebooks" / "LLMOps_Assignment2.ipynb").exists())

.venv  -  Directory
data  -  Directory
mlartifacts  -  Directory
mlflow.db  -  File
notebooks  -  Directory
outputs  -  Directory
README.md  -  File
reports  -  Directory
requirements.txt  -  File
screenshots  -  Directory

Requirements file exists: True
README file exists: True
Notebook exists: True


In [18]:

# Final cost calculation using exact model-specific Featherless pricing

INPUT_PRICE = 0.04
OUTPUT_PRICE = 0.08

results_df["input_cost_usd"] = (
    results_df["input_tokens"] / 1_000_000
) * INPUT_PRICE

results_df["output_cost_usd"] = (
    results_df["output_tokens"] / 1_000_000
) * OUTPUT_PRICE

results_df["estimated_hosted_cost_usd"] = (
    results_df["input_cost_usd"] +
    results_df["output_cost_usd"]
)

results_df["local_ollama_api_cost_usd"] = 0.0
results_df["pricing_provider"] = "Featherless AI - model-specific listing"
results_df["pricing_date"] = "2026-11-05"
results_df["currency"] = "USD"

prompt_comparison = (
    results_df.groupby("prompt_version")
    .agg(
        avg_latency_seconds=("latency_seconds", "mean"),
        avg_input_tokens=("input_tokens", "mean"),
        avg_output_tokens=("output_tokens", "mean"),
        avg_total_tokens=("total_tokens", "mean"),
        avg_cost_per_request=("estimated_hosted_cost_usd", "mean"),
        valid_json=("valid_json", "mean")
    )
    .reset_index()
)

prompt_comparison["monthly_projection_10000_usd"] = (
    prompt_comparison["avg_cost_per_request"] * 10000
)

# Update existing MLflow runs
from mlflow.tracking import MlflowClient

client = MlflowClient()

for item in mlflow_runs:
    version = item["prompt_version"]

    new_cost = prompt_comparison.loc[
        prompt_comparison["prompt_version"] == version,
        "avg_cost_per_request"
    ].iloc[0]

    client.log_metric(
        run_id=item["run_id"],
        key="avg_cost_per_request",
        value=float(new_cost)
    )

    item["avg_cost_per_request"] = float(new_cost)

# Save the final corrected files
results_df.to_csv(
    OUTPUT_DIR / "latency_tokens_cost.csv", index=False
)

prompt_comparison.to_csv(
    OUTPUT_DIR / "prompt_comparison.csv", index=False
)

prompt_evaluation_df = pd.DataFrame(mlflow_runs)

prompt_evaluation_df.to_csv(
    OUTPUT_DIR / "prompt_evaluation_results.csv",
    index=False
)

# Correct the README pricing details
readme_path = PROJECT_DIR / "README.md"
readme = readme_path.read_text(encoding="utf-8")

readme = readme.replace(
    "Input: USD 0.01 per million tokens",
    "Input: USD 0.04 per million tokens"
)
readme = readme.replace(
    "Output: USD 0.02 per million tokens",
    "Output: USD 0.08 per million tokens"
)

readme_path.write_text(readme, encoding="utf-8")

print("FINAL MODEL-SPECIFIC PRICING APPLIED")
display(
    prompt_comparison[
        [
            "prompt_version",
            "avg_cost_per_request",
            "valid_json",
            "monthly_projection_10000_usd"
        ]
    ].round(10)
)


FINAL MODEL-SPECIFIC PRICING APPLIED


,prompt_version,avg_cost_per_request,valid_json,monthly_projection_10000_usd
0,v1,0.000023,0.0,0.22832
1,v2,0.000024,0.0,0.23728
2,v3,0.000022,0.8,0.21888


In [19]:

# Final prompt comparison table for assignment submission

final_comparison = prompt_comparison.copy()

final_comparison["valid_json_rate"] = (
    final_comparison["valid_json"] * 100
).round(1).astype(str) + "%"

final_comparison["recommended"] = (
    final_comparison["prompt_version"] == "v3"
).map({True: "Yes", False: "No"})

final_comparison = final_comparison.rename(columns={
    "prompt_version": "Prompt Version",
    "avg_latency_seconds": "Avg Latency (seconds)",
    "avg_input_tokens": "Avg Input Tokens",
    "avg_output_tokens": "Avg Output Tokens",
    "avg_cost_per_request": "Avg Cost (USD)",
    "valid_json_rate": "Valid JSON Rate",
    "recommended": "Recommended?"
})

final_comparison = final_comparison[
    [
        "Prompt Version",
        "Avg Latency (seconds)",
        "Avg Input Tokens",
        "Avg Output Tokens",
        "Avg Cost (USD)",
        "Valid JSON Rate",
        "Recommended?"
    ]
]

display(final_comparison.round(9))

final_comparison.to_csv(
    OUTPUT_DIR / "prompt_comparison.csv",
    index=False
)


,Prompt Version,Avg Latency (seconds),Avg Input Tokens,Avg Output Tokens,Avg Cost (USD),Valid JSON Rate,Recommended?
0,v1,15.957504,46.4,262.2,0.000023,0.0%,No
1,v2,15.944290,68.4,262.4,0.000024,0.0%,No
2,v3,13.806184,102.4,222.4,0.000022,80.0%,Yes


In [20]:
display(final_comparison.round(9).style.hide(axis="index"))

Prompt Version,Avg Latency (seconds),Avg Input Tokens,Avg Output Tokens,Avg Cost (USD),Valid JSON Rate,Recommended?
v1,15.957504,46.400000,262.200000,0.000023,0.0%,No
v2,15.944290,68.400000,262.400000,0.000024,0.0%,No
v3,13.806184,102.400000,222.400000,0.000022,80.0%,Yes


In [21]:

# Task 1: Latency summary statistics

import pandas as pd
from pathlib import Path

# Calculate statistics separately for each prompt version
latency_summary = (
    results_df.groupby("prompt_version")["latency_seconds"]
    .agg(
        mean="mean",
        median="median",
        minimum="min",
        maximum="max",
        standard_deviation="std"
    )
    .reset_index()
)

# Calculate overall statistics across all 15 requests
overall = pd.DataFrame([{
    "prompt_version": "overall",
    "mean": results_df["latency_seconds"].mean(),
    "median": results_df["latency_seconds"].median(),
    "minimum": results_df["latency_seconds"].min(),
    "maximum": results_df["latency_seconds"].max(),
    "standard_deviation": results_df["latency_seconds"].std()
}])

latency_summary = pd.concat(
    [latency_summary, overall],
    ignore_index=True
)

# Display results
print("LATENCY SUMMARY STATISTICS (SECONDS)")
display(latency_summary.round(4))

# Save CSV
latency_summary.to_csv(
    OUTPUT_DIR / "latency_statistics.csv",
    index=False
)

print("Latency statistics saved successfully.")


LATENCY SUMMARY STATISTICS (SECONDS)


,prompt_version,mean,median,minimum,maximum,standard_deviation
0,v1,15.9575,13.6961,7.4740,29.3446,8.1732
1,v2,15.9443,15.0571,12.3406,19.7226,3.0149
2,v3,13.8062,10.7143,5.9820,29.8436,9.3181
3,overall,15.2360,13.6961,5.9820,29.8436,6.8982


Latency statistics saved successfully.


In [22]:

# Identify and analyse the slowest request

slowest = results_df.loc[
    results_df["latency_seconds"].idxmax()
]

print("SLOWEST REQUEST")
print("----------------")
print("Question ID:", slowest["question_id"])
print("Prompt version:", slowest["prompt_version"])
print("Model:", slowest["model_name"])
print("Latency:", round(slowest["latency_seconds"], 4), "seconds")
print("Output tokens:", slowest["output_tokens"])
print("Stop reason:", slowest["done_reason"])
print("Response status:", slowest["response_status"])


SLOWEST REQUEST
----------------
Question ID: Q2
Prompt version: v3
Model: qwen2.5:1.5b
Latency: 29.8436 seconds
Output tokens: 512
Stop reason: length
Response status: 200


### Latency Interpretation

The slowest request was Q2 using Prompt Version 3, which took
approximately 29.84 seconds. The request generated 512 output tokens
and stopped because it reached the configured generation limit.

Longer response generation increases inference time. Local hardware
resources, model processing speed and the complexity of the question
may also influence latency.

Version 3 achieved the lowest average latency of approximately
13.81 seconds, compared with 15.96 seconds for v1 and 15.94 seconds
for v2. However, its response-time variability was higher.

In [23]:

# Task 1: Five-question latency table

task1_latency_table = results_df[
    results_df["prompt_version"] == "v1"
][
    [
        "question_id",
        "prompt_version",
        "model_name",
        "start_time",
        "end_time",
        "latency_seconds",
        "response_status"
    ]
].copy()

display(task1_latency_table.round(4))

task1_latency_table.to_csv(
    OUTPUT_DIR / "task1_latency_five_questions.csv",
    index=False
)

print("Five-question latency table saved.")


,question_id,prompt_version,model_name,start_time,end_time,latency_seconds,response_status
0,Q1,v1,qwen2.5:1.5b,2026-11-05T14:31:48.201260+00:00,2026-11-05T14:31:55.675284+00:00,7.4740,200
1,Q2,v1,qwen2.5:1.5b,2026-11-05T14:31:56.783859+00:00,2026-11-05T14:32:09.494783+00:00,12.7109,200
2,Q3,v1,qwen2.5:1.5b,2026-11-05T14:32:10.612072+00:00,2026-11-05T14:32:27.173928+00:00,16.5618,200
3,Q4,v1,qwen2.5:1.5b,2026-11-05T14:32:28.275418+00:00,2026-11-05T14:32:57.620040+00:00,29.3446,200
4,Q5,v1,qwen2.5:1.5b,2026-11-05T14:32:58.699505+00:00,2026-11-05T14:33:12.395631+00:00,13.6961,200


Five-question latency table saved.
